# **MLIS Practical 03**

**Name:** Yesha Pandya  
**Enrollment No:** 23BT04175  
**Division:** A
**Batch**: C

## **Objective:**
Implement and understand K Nearnest Neighbour (KNN) Classification Algorithm

# **1. Theory + Concepts**
### **1.1 Overview of KNN**
K-Nearest Neighbors (KNN) is a simple, non-parametric and lazy supervised machine learning algorithm used for both classification and regression tasks.

* Lazy Learner (Instance-Based):
  * KNN does not explicitly build a model or learn a discriminative function during the training phase. Instead, it simply stores the training dataset.
  * All computations (distance calculations and voting) take place during the query/testing phase.

* Non-Parametric:
  * It makes no explicit assumptions about the underlying statistical distribution of the dataset (e.g., normality)
  * This makes it highly flexible for non-linear decision boundaries.

### **1.2 Working Principle**
When a new test instance $x$ is provided:The algorithm computes the distance between $x$ and every data point in the training set.It ranks the distances in ascending order and selects the top $K$ closest training samples.For Classification: It applies a majority voting rule among the $K$ neighbors. The predicted class is the class label that appears most frequently among the $K$ nearest neighbors.For Regression: It returns the average (mean) value of the targets of the $K$ nearest neighbors.

## **2. Mathematical Formulation**
### **2.1 Distance Metrics**
The core of KNN relies on measuring the distance/dissimilarity between feature vectors in an $n$-dimensional vector space.

#### **1. Euclidean Distance (L2 Norm)**
The most commonly used distance metric for continuous variables:
$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

#### **2. Manhattan Distance (L1 Norm / City Block)**
Calculates distance measured along axes at right angles:
$$d(x, y) = \sum_{i=1}^{n} \vert{}x_i - y_i\vert{}$$

#### **3. Minkowski Distance**
A generalized metric encompassing both Euclidean ($p=2$) and Manhattan ($p=1$) distances:
$$d(x, y) = \left( \sum_{i=1}^{n} \vert{}x_i - y_i\vert{}^p \right)^{\frac{1}{p}}$$

### **2.2 Hyperparameter Selection: Choice of $K$**
The parameter $K$ controls the trade-off between bias and variance:
* Small $K$ (e.g., $K = 1$):
  * High variance, low bias.
  * Highly sensitive to noise and outliers in the dataset.
  * Leads to overfitting (complex decision boundaries).

* Large $K$:
  * High bias, low variance.
  * Smooths out noise but may include samples from neighboring classes.
  * Leads to underfitting (oversimplified decision boundaries).
  
**Best Practices for Choosing $K$:**
* Set $K = \sqrt{N}$, where $N$ is the total number of samples in the training set.
* Use an odd value of $K$ for binary classification to avoid tie votes.
* Determine the optimal $K$ value empirically using cross-validation.

## **3. Algorithm Procedure**

```
Algorithm: K-Nearest Neighbors Classification

Inputs:
  - Training dataset: D = {(x1, y1), (x2, y2), ..., (xN, yN)}
  - Test instance: x_test
  - Number of neighbors: K

Steps:
1. For each training sample (xi, yi) in D:
      a. Calculate the Euclidean distance d(x_test, xi).
2. Sort all calculated distances in ascending order.
3. Select the top K training samples corresponding to the smallest distances.
4. Count the frequency of each class label among these K neighbors.
5. Assign x_test the class label with the highest count (mode).
```

## **NOTE**
### **Advantages**
* Simplicity: Easy to understand, mathematical intuition is straightforward, and simple to implement.
* Zero Training Time: As a lazy learner, there is no training overhead.
* Non-Parametric: Handles arbitrary decision boundaries effectively.Naturally
* Multi-Class: Works natively for multi-class classification problems without modifications.

### **Disadvantages**
* Computational Cost at Test Time: Query time grows linearly $O(N \cdot D)$ with dataset size $\implies$ makes it slow for large datasets.
* Sensitivity to Feature Scaling: Features measured in larger units dominate distance metrics. Feature scaling (Min-Max Normalization / StandardScaler) is required.
* Curse of Dimensionality: In high-dimensional spaces, distance metrics become less meaningful as all points tend to become equidistant.
* Memory Intensive: Must retain the entire training dataset in memory.

## **4. Implementation**

### **4.1 Implementation from scratch (using NumPy and Python)**

In [1]:
import numpy as np
from collections import Counter

class KNNClassifierScratch:
    def __init__(self, k=3):
        """
        Initialize KNN classifier.
        :param k: Number of nearest neighbors (default=3)
        """
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        """
        Store the training instances (Lazy learning).
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _euclidean_distance(self, x1, x2):
        """
        Compute Euclidean distance between vector x1 and vector x2.
        """
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def _predict_single(self, x):
        """
        Predict label for a single query instance.
        """
        # Step 1: Calculate distances to all training samples
        distances = [self._euclidean_distance(x, x_train) for x_train in self.X_train]

        # Step 2: Find indices of top K smallest distances
        k_indices = np.argsort(distances)[:self.k]

        # Step 3: Extract corresponding class labels
        k_nearest_labels = [self.y_train[i] for i in k_indices]

        # Step 4: Majority vote (mode)
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

    def predict(self, X):
        """
        Predict labels for an array of test instances.
        """
        X = np.array(X)
        return np.array([self._predict_single(x) for x in X])


# Driver Code / Execution Example
if __name__ == "__main__":
    # Synthetic Dataset: [Feature 1, Feature 2]
    X_train = np.array([
        [1.0, 2.0],
        [1.5, 1.8],
        [5.0, 8.0],
        [6.0, 9.0],
        [1.2, 0.9],
        [5.5, 8.5]
    ])
    # Class labels: 0 or 1
    y_train = np.array([0, 0, 1, 1, 0, 1])

    # Test Samples
    X_test = np.array([
        [1.1, 1.5],  # Should be Class 0
        [5.2, 8.2]   # Should be Class 1
    ])

    # Initialize and fit custom model
    knn = KNNClassifierScratch(k=3)
    knn.fit(X_train, y_train)

    # Predict
    predictions = knn.predict(X_test)

    # Display Results
    print("Custom KNN Implementation Output:")
    for idx, test_point in enumerate(X_test):
        print(f"Sample: {test_point} -> Predicted Class: {predictions[idx]}")

Custom KNN Implementation Output:
Sample: [1.1 1.5] -> Predicted Class: 0
Sample: [5.2 8.2] -> Predicted Class: 1


### **4.2 Implementation using `scikit-learn`**

In [3]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# Synthetic Dataset
X_train = np.array([[1.0, 2.0], [1.5, 1.8], [5.0, 8.0], [6.0, 9.0], [1.2, 0.9], [5.5, 8.5]])
y_train = np.array([0, 0, 1, 1, 0, 1])

X_test = np.array([[1.1, 1.5], [5.2, 8.2]])

# Initialize Scikit-Learn KNN Classifier
model = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

print("Scikit-Learn KNN Output")
print("Predictions:", predictions)

Scikit-Learn KNN Output
Predictions: [0 1]


## **Conclusion**
In this practical, the K-Nearest Neighbors (KNN) algorithm was successfully implemented both from scratch using standard Python/NumPy structures and using scikit-learn.

The algorithm classifies test points based on distance metrics and majority voting among $K$ neighbors. Feature scaling and proper choice of $K$ are crucial for achieving optimal classification performance